# Timing Benchmarks: GULPS Scaling Analysis

This notebook establishes that GULPS discrete ISA compilation time scales **O(n)** with circuit depth.

## Approach
1. Build multiple discrete ISAs with different gate combinations
2. For each ISA, decompose N=1000 random unitaries end-to-end
3. Track **empirical metrics** from returned circuits: depth, cost, fidelity, duration
4. Plot: median numeric time vs median empirical depth
5. Fit O(n) linear model to demonstrate scaling

This pattern will extend cleanly to NUOP and BQSKit comparisons.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import trange

import scienceplots
import lovelyplots

from qiskit.quantum_info import Operator, average_gate_fidelity, random_unitary

from gulps import GulpsDecomposer, logger
from gulps.core.isa import DiscreteISA

from benchmark_isas import get_benchmark_isas

logger.setLevel("WARNING")

## Part 1: Load ISA Configurations

In [ ]:
# Load ISA configurations from helper file
isa_configs = get_benchmark_isas()
print(f"Loaded {len(isa_configs)} ISA configurations")

## Part 2: Empirical Benchmarks

For each ISA, decompose N random unitaries end-to-end using `decomposer(u)`.

In [ ]:
N_SAMPLES = 1000
SEED_OFFSET = 0

results = []

for name, gate_set in isa_configs:
    print(f"\n{'='*60}")
    print(f"Benchmarking: {name}")
    print(f"{'='*60}")
    
    # Build ISA (no precompute_polytopes for speed)
    gates, costs, labels = zip(*gate_set)
    isa = DiscreteISA(
        gate_set=gates,
        costs=costs,
        names=labels,
        precompute_polytopes=False,
        single_qubit_cost=1e-6,
    )
    decomposer = GulpsDecomposer(isa=isa)
    
    # Warm-up (JIT compilation)
    _ = decomposer(random_unitary(4, seed=999))
    
    # Collect empirical data
    fidelities = []
    numeric_times = []
    total_times = []
    circuit_depths = []  # Count 2Q gates in circuit
    circuit_costs = []   # Sum gate costs
    failures = 0
    
    for idx in trange(N_SAMPLES, desc=f"Decomposing", leave=False):
        u = random_unitary(4, seed=SEED_OFFSET + idx)
        
        try:
            # End-to-end decomposition
            circuit = decomposer(u)
            v = Operator(circuit)
            
            # Fidelity
            fid = average_gate_fidelity(u, v)
            fidelities.append(fid)
            
            # Timing (tracked internally)
            numeric_times.append(decomposer.last_timing["numeric"])
            total_times.append(decomposer.last_timing["total"])
            
            # Empirical depth: count 2Q gates
            depth = sum(1 for instr in circuit.data if len(instr.qubits) == 2)
            circuit_depths.append(depth)
            
            # Empirical cost: sum gate costs
            cost = 0.0
            for instr in circuit.data:
                if len(instr.qubits) == 2:
                    # Match gate name to ISA label
                    gate_name = instr.operation.name
                    for i, label in enumerate(labels):
                        if label in gate_name or gate_name in label:
                            cost += costs[i]
                            break
                else:
                    # Single-qubit gates
                    cost += isa.single_qubit_cost
            circuit_costs.append(cost)
            
        except Exception as e:
            fidelities.append(np.nan)
            numeric_times.append(np.nan)
            total_times.append(np.nan)
            circuit_depths.append(np.nan)
            circuit_costs.append(np.nan)
            failures += 1
    
    # Convert to arrays
    fidelities = np.array(fidelities)
    numeric_times = np.array(numeric_times)
    total_times = np.array(total_times)
    circuit_depths = np.array(circuit_depths)
    circuit_costs = np.array(circuit_costs)
    
    # Compute statistics (excluding NaNs)
    valid_mask = ~np.isnan(fidelities)
    
    median_fidelity = np.median(fidelities[valid_mask])
    median_numeric_time = np.median(numeric_times[valid_mask])
    median_total_time = np.median(total_times[valid_mask])
    median_depth = np.median(circuit_depths[valid_mask])
    median_cost = np.median(circuit_costs[valid_mask])
    
    std_numeric_time = np.std(numeric_times[valid_mask])
    std_depth = np.std(circuit_depths[valid_mask])
    
    results.append({
        "name": name,
        "fidelities": fidelities,
        "numeric_times": numeric_times,
        "total_times": total_times,
        "circuit_depths": circuit_depths,
        "circuit_costs": circuit_costs,
        "failures": failures,
        "median_fidelity": median_fidelity,
        "median_numeric_time": median_numeric_time,
        "median_total_time": median_total_time,
        "median_depth": median_depth,
        "median_cost": median_cost,
        "std_numeric_time": std_numeric_time,
        "std_depth": std_depth,
    })
    
    # Progress report
    print(f"  Median fidelity:      {median_fidelity:.6f}")
    print(f"  Median numeric time:  {median_numeric_time:.4f}s (±{std_numeric_time:.4f}s)")
    print(f"  Median depth:         {median_depth:.2f} (±{std_depth:.2f})")
    print(f"  Median cost:          {median_cost:.3f}")
    print(f"  Failures:             {failures}/{N_SAMPLES}")

print(f"\n\n{'='*60}")
print(f"✓ Completed benchmarks for {len(results)} ISAs")
print(f"{'='*60}")

## Part 3: O(n) Scaling Analysis

Plot median numeric time vs median empirical depth with linear fit.

In [ ]:
# Extract data for plotting
x_vals = np.array([r["median_depth"] for r in results])
y_vals = np.array([r["median_numeric_time"] for r in results])
y_err = np.array([r["std_numeric_time"] for r in results])
names = [r["name"] for r in results]

# Remove any invalid entries
valid_idx = ~(np.isnan(x_vals) | np.isnan(y_vals))
x_vals = x_vals[valid_idx]
y_vals = y_vals[valid_idx]
y_err = y_err[valid_idx]
names = [n for n, v in zip(names, valid_idx) if v]

print(f"Plotting {len(x_vals)} ISAs")

In [ ]:
with plt.style.context(["science", "ieee"]):
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Color and marker schemes
    colors = cm.tab20(np.linspace(0, 1, len(x_vals)))
    markers = ["o", "s", "^", "D", "v", "p", "*", "<", ">", "h"]
    
    # Scatter plot with error bars
    for i, (x, y, yerr, name) in enumerate(zip(x_vals, y_vals, y_err, names)):
        # Error bars clipped at 0
        err_low = y - max(y - yerr, 0)
        err_high = y + yerr - y
        
        ax.errorbar(
            x, y,
            yerr=[[err_low], [err_high]],
            fmt=markers[i % len(markers)],
            color=colors[i],
            markersize=8,
            capsize=4,
            elinewidth=1.5,
            label=name,
            alpha=0.8,
        )
    
    # O(n) linear fit
    if len(x_vals) > 1:
        coeffs = np.polyfit(x_vals, y_vals, 1)
        fit_fn = np.poly1d(coeffs)
        x_fit = np.linspace(x_vals.min(), x_vals.max(), 100)
        
        ax.plot(
            x_fit,
            fit_fn(x_fit),
            "--",
            color="red",
            linewidth=2.5,
            label=f"O(n) fit: {coeffs[0]:.4f}n + {coeffs[1]:.4f}",
            alpha=0.7,
            zorder=0,
        )
        
        # Compute R² for goodness of fit
        y_pred = fit_fn(x_vals)
        ss_res = np.sum((y_vals - y_pred) ** 2)
        ss_tot = np.sum((y_vals - np.mean(y_vals)) ** 2)
        r_squared = 1 - (ss_res / ss_tot)
        
        ax.text(
            0.05, 0.95,
            f"$R^2 = {r_squared:.4f}$",
            transform=ax.transAxes,
            fontsize=12,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
        )
    
    ax.set_xlabel("Median Circuit Depth (# 2Q gates)")
    ax.set_ylabel("Median Numeric Time (s)")
    ax.set_title("GULPS Discrete ISAs: O(n) Scaling with Circuit Depth")
    ax.legend(fontsize=8, frameon=True, loc="best", ncol=2)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Summary Table

In [ ]:
import pandas as pd

# Create summary dataframe
summary_data = {
    "ISA": [r["name"] for r in results],
    "Median Depth": [r["median_depth"] for r in results],
    "Median Cost": [r["median_cost"] for r in results],
    "Median Time (s)": [r["median_numeric_time"] for r in results],
    "Median Fidelity": [r["median_fidelity"] for r in results],
    "Failures": [f"{r['failures']}/{N_SAMPLES}" for r in results],
}

df = pd.DataFrame(summary_data)
df = df.sort_values("Median Depth")

print("\n" + "="*80)
print("SUMMARY: GULPS Discrete ISA Benchmarks")
print("="*80)
print(df.to_string(index=False))
print("="*80)